## we'll implement the last step of the Rag technique, the generation Step.
### We'll create an expression language chain from scratch that uses the documents returned by the retriever to generate a context specific respons

In [1]:
import config

from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_chroma import Chroma  # New way
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import RunnableParallel
from langchain_core.output_parsers import StrOutputParser

C:\Users\moham\anaconda3\envs\Langchain_env\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
embedding = OpenAIEmbeddings(model="text-embedding-ada-002",
                            openai_api_key = config.api_key)
vectorstore = Chroma( persist_directory= './vectore representrtion folder',
                                   embedding_function= embedding
                                   )

In [3]:
len(vectorstore.get()['documents'])

22

In [4]:
retriver = vectorstore.as_retriever(search_type ='mmr',
                                   search_kwargs={'k':3 , 'lambda':0.7})

In [5]:
#Let's now define our prompt template from a string template.
#At the beginning of the string, we require the user's question.
#Next should come the context from the retriever.
#Finally, we end the template with instructions to specify the resource.

TEMPLATE = '''
Answer the following question:
{question}

To answer the question, use only the following context:
{context}

At the end of the response, specify the name of the lecture this context is taken from in the format:
Resources: *Lecture Title*
where *Lecture Title* should be substituted with the title of all resource lectures.
'''

prompt_template = PromptTemplate.from_template(TEMPLATE)

In [6]:
chat = ChatOpenAI(
    model = "gpt-3.5-turbo",
    temperature = 0,
    seed = 365,
    max_tokens = 300,
    openai_api_key = config.api_key
)

In [7]:
question = "What software do data scientists use?"

## We have all the components ready to construct our chain.



In [16]:
chain = RunnableParallel({
            'context' : retriver,
    'question' : RunnablePassthrough()
})

In [17]:
chain.invoke(question)

{'context': [Document(id='1f473953-d596-4da7-8265-df4fe1ad083e', metadata={'Course Title': 'INTRODUCTION', 'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need'}, page_content='As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they can manipulate data and are integrated within multiple data and data science software platforms. They are not just suitable for mathematical and statistical computations. In other words, R, and Python are adaptable. They can solve a wide variety of business and data-related problems from beginning to the end'),
  Document(id='536dc62b-aa6d-4aba-aa10-66af8defc5d1', metadata={'Course Title': 'INTRODUCTION', 'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need'}, page_content='Among the many applications we have plotted, we can say there is an increasing amount of software designed fo

In [18]:
chain1 = {
            'context' : retriver,
    'question' : RunnablePassthrough()
} | prompt_template

The input variables, question and context are successfully substituted with the respective texts.

In [19]:
chain1.invoke(question)

StringPromptValue(text="\nAnswer the following question:\nWhat software do data scientists use?\n\nTo answer the question, use only the following context:\n[Document(id='1f473953-d596-4da7-8265-df4fe1ad083e', metadata={'Course Title': 'INTRODUCTION', 'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need'}, page_content='As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they can manipulate data and are integrated within multiple data and data science software platforms. They are not just suitable for mathematical and statistical computations. In other words, R, and Python are adaptable. They can solve a wide variety of business and data-related problems from beginning to the end'), Document(id='536dc62b-aa6d-4aba-aa10-66af8defc5d1', metadata={'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need', 'Course Title'

In [21]:
print("\nAnswer the following question:\nWhat software do data scientists use?\n\nTo answer the question, use only the following context:\n[Document(id='1f473953-d596-4da7-8265-df4fe1ad083e', metadata={'Course Title': 'INTRODUCTION', 'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need'}, page_content='As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they can manipulate data and are integrated within multiple data and data science software platforms. They are not just suitable for mathematical and statistical computations. In other words, R, and Python are adaptable. They can solve a wide variety of business and data-related problems from beginning to the end'), Document(id='536dc62b-aa6d-4aba-aa10-66af8defc5d1', metadata={'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need', 'Course Title': 'INTRODUCTION'}, page_content='Among the many applications we have plotted, we can say there is an increasing amount of software designed for working with big data such as Apache Hadoop, Apache Hbase, and Mongo DB. In terms of big data, Hadoop is the name that must stick with you. Hadoop is listed as a software in the sense that it is a collection of programs, but don’t imagine it as a nice-looking application'), Document(id='8d549176-6080-4b0f-89b2-a2f5b367c462', metadata={'Course Title': 'INTRODUCTION', 'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need'}, page_content='Great! We hope we gave you a good idea about the level of applicability of the most frequently used programming and software tools in the field of data science. Thank you for watching!')]\n\nAt the end of the response, specify the name of the lecture this context is taken from in the format:\nResources: *Lecture Title*\nwhere *Lecture Title* should be substituted with the title of all resource lectures.\n")



Answer the following question:
What software do data scientists use?

To answer the question, use only the following context:
[Document(id='1f473953-d596-4da7-8265-df4fe1ad083e', metadata={'Course Title': 'INTRODUCTION', 'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need'}, page_content='As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they can manipulate data and are integrated within multiple data and data science software platforms. They are not just suitable for mathematical and statistical computations. In other words, R, and Python are adaptable. They can solve a wide variety of business and data-related problems from beginning to the end'), Document(id='536dc62b-aa6d-4aba-aa10-66af8defc5d1', metadata={'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need', 'Course Title': 'INTRODUCTION'}, page_conte

In [22]:
chain2 = ({
            'context' : retriver,
    'question' : RunnablePassthrough()
} | prompt_template | chat)

In [23]:
chain2.invoke(question)

AIMessage(content='Data scientists use R and Python as the two most popular tools in data science. These tools are adaptable and can solve a wide variety of business and data-related problems. Additionally, software designed for working with big data such as Apache Hadoop, Apache Hbase, and Mongo DB are also commonly used in data science.\n\nResources: Programming Languages & Software Employed in Data Science - All the Tools You Need', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 80, 'prompt_tokens': 466, 'total_tokens': 546, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DPoQBbMEnylsBlT3VQkPQskUUzHgV', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id=

In [24]:
print("Data scientists use R and Python as the two most popular tools in data science. These tools are adaptable and can solve a wide variety of business and data-related problems. Additionally, software designed for working with big data such as Apache Hadoop, Apache Hbase, and Mongo DB are also commonly used in data science.\n\nResources: Programming Languages & Software Employed in Data Science - All the Tools You Need")

Data scientists use R and Python as the two most popular tools in data science. These tools are adaptable and can solve a wide variety of business and data-related problems. Additionally, software designed for working with big data such as Apache Hadoop, Apache Hbase, and Mongo DB are also commonly used in data science.

Resources: Programming Languages & Software Employed in Data Science - All the Tools You Need


## Indeed, the model considers the retrieved context and has also remembered to include the resource.

What's left is to introduce one final component, an instance of the string output parser class, which

converts this AI message object to a simple string.



In [25]:
chain3 = ({
            'context' : retriver,
    'question' : RunnablePassthrough()
} | prompt_template | chat |StrOutputParser())

In [26]:
chain3.invoke(question)

'Data scientists use R and Python as the two most popular tools across all columns. These tools can manipulate data and are integrated within multiple data and data science software platforms, making them adaptable for solving a wide variety of business and data-related problems. Additionally, software designed for working with big data such as Apache Hadoop, Apache Hbase, and Mongo DB are also commonly used in data science.\nResources: Programming Languages & Software Employed in Data Science - All the Tools You Need'

## STUFFING inserting all retrieved documents into the prompt is called stuffing.
but it wouldn't be a suitable choice when, for example,

the number of documents or their content is so extensive that it doesn't fit the context window limit

of the model.

As an additional drawback, research shows that models primarily use information at the beginning or

end of a document list.

Therefore, information in the middle of the retrieved list may be disregarded by the model, regardless

of its importance.

One method alternative to stuffing is document refinement.

This technique solves the problem of context window limits by passing the documents to the model one

at a time.

When an answer is generated, it's passed to the model with the next document in the list and the developer's

instructions, thereby updating the old response with the additional information it has received.

When all documents have been fed to the model, the answer should account for each piece of information.

Of course, contrary to the stuffing technique, this method comes at the cost of passing many more

calls to the LM, making it more expensive.